# Air pollution in Karaganda: exploratory analysis

Case Study Task 1. Descriptive analysis only, no machine learning: this notebook reproduces the
numbers and figures in `report.md`.

Run it from the `case-study-1` folder.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'case-study-1' else Path.cwd()
KAZ = ROOT / 'data' / 'raw' / 'kazhydromet'
pd.set_option('display.width', 140)
print('project root:', ROOT)

## 1. How bad is Karaganda compared with other cities?

In [ ]:
rank = pd.read_csv(KAZ / 'national' / 'kz_city_ranking.csv')
print(f'{len(rank)} settlements ranked by Kazhydromet, H1 2026')
print(f'Karaganda: rank {rank["rank"].iloc[0]}, '
      f'score {rank.score.iloc[0]}, {rank.vz_cases_H1_2026.iloc[0]} episodes '
      f'out of {rank.vz_cases_H1_2026.sum()} nationwide '
      f'({100 * rank.vz_cases_H1_2026.iloc[0] / rank.vz_cases_H1_2026.sum():.0f}%)')
rank.head(10)[['rank', 'city', 'score', 'vz_cases_H1_2026', 'airkaz_sensors']]

## 2. Which pollutants exceed the limits?

`ratio_pdk_ss` is the measured concentration divided by the Kazakh 24-hour permissible limit, so
a value above 1.0 means the limit was exceeded on average over the quarter.

In [ ]:
series = pd.read_csv(KAZ / 'krg_series.csv')
quarters = series[series.granularity == 'quarter']

ranking = (quarters.groupby('pollutant')
           .agg(mean_ratio=('ratio_pdk_ss', 'mean'),
                worst_single=('ratio_pdk_mr', 'max'),
                share_above=('np_pct', 'mean'))
           .sort_values('mean_ratio', ascending=False))
ranking.round(2).head(10)

PM2.5 is 5.6 times the limit on average and nothing else is close. Note that sulphur dioxide,
nitrogen dioxide and carbon monoxide stay below their limits: the problem is particles from
low-level sources, not gases from tall stacks.

## 3. Seasonality: the quarterly measurements

In [ ]:
pm = quarters[quarters.pollutant == 'PM2.5'].copy()
pm['quarter'] = pm.period.str[-1]
pm['ug_m3'] = pm.mean_mgm3 * 1000                       # the bulletins report mg/m3
print(pm.groupby('quarter').ug_m3.mean().round(0).rename('mean PM2.5, ug/m3').to_string())
print()
pm[['period', 'ug_m3', 'ratio_pdk_ss', 'np_pct']].round(2).to_string(index=False)

Quarters 1 and 4, the heating season, run 1.5 to 2 times higher than quarters 2 and 3.

## 4. Seasonality: the measured episode record

These are the days on which Kazhydromet actually registered high pollution.

In [ ]:
episodes = pd.read_csv(KAZ / 'krg_vz_days.csv', parse_dates=['date'])
by_month = episodes.date.dt.month.value_counts().reindex(range(1, 13), fill_value=0)
by_month.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(f'{len(episodes)} episode days, 2021-2025')
print(by_month.rename('days').to_string())
print()
print('by year: ', episodes.date.dt.year.value_counts().sort_index().to_dict())
print('by post: ', episodes.posts.value_counts().to_dict())

Every episode falls between September and April, and all of them come from posts 6 and 8 in the
Prishakhtinsk district. Since 2024 only post 8 appears.

## 5. What weather produces an episode?

Restricted to heating-season days, so the comparison reflects the weather and not the calendar.

In [ ]:
daily = pd.read_csv(ROOT / 'data/raw/open_meteo/karaganda_daily.csv', parse_dates=['date'])
heating = daily[(daily.date.dt.year <= 2025) & daily.date.dt.month.isin([1, 2, 3, 10, 11, 12])]

cols = ['temperature_2m_mean', 'wind_speed_10m_mean', 'blh_min_m', 'surface_pressure_mean']
table = heating.groupby('vz_day')[cols].median().T
table.columns = ['ordinary day', 'episode day']
print(f'{int(heating.vz_day.sum())} episode days vs {int((heating.vz_day == 0).sum())} ordinary days')
table.round(1)

Wind speed halves and the mixing layer collapses from 135 to 30 metres. Frost, calm and high
pressure: a classic temperature inversion.

In [ ]:
# correlation of PM2.5 with weather, whole record and split by season
cams = daily[daily.cams_pm2_5.notna()].copy()
cams['heating'] = cams.date.dt.month.isin([1, 2, 3, 10, 11, 12])

for label, sub in [('all year', cams), ('heating', cams[cams.heating]), ('warm', cams[~cams.heating])]:
    r = sub[['wind_speed_10m_mean', 'blh_min_m', 'temperature_2m_mean']].corrwith(sub.cams_pm2_5)
    print(f'{label:9s} (n={len(sub):4d})  ' + '  '.join(f'{k.split("_")[0]} {v:+.2f}' for k, v in r.items()))

Both dispersion variables roughly double in strength during the heating season.

## 6. Comparison with the WHO guidelines

WHO 2021: 5 ug/m3 as an annual mean, 15 ug/m3 as a 24-hour mean.

In [ ]:
WHO_ANNUAL, WHO_DAILY = 5.0, 15.0

print('--- reanalysis, the conservative estimate ---')
annual = cams.groupby(cams.date.dt.year).cams_pm2_5.mean()
for year, v in annual.items():
    print(f'  {year}: {v:5.2f} ug/m3   {v / WHO_ANNUAL:.1f}x the WHO annual guideline')
print(f'  days above the 24-hour guideline: {(cams.cams_pm2_5 > WHO_DAILY).sum()} of {len(cams)} '
      f'({100 * (cams.cams_pm2_5 > WHO_DAILY).mean():.1f}%)')

print()
print('--- ground measurements, the high estimate ---')
for _, row in pm.iterrows():
    print(f'  {row.period}: {row.ug_m3:5.0f} ug/m3   {row.ug_m3 / WHO_ANNUAL:.0f}x the WHO annual guideline')

The two sources disagree by a factor of about 20. The 40 km reanalysis cell averages the city
with empty steppe and must underestimate; the ground posts sit next to coal-burning housing and
may overstate a city-wide mean. The honest reading is a range: **6 to 30 times the WHO guideline**.

## 7. Where the emissions come from

In [ ]:
sources = pd.read_csv(KAZ / 'krg_sources.csv')
print(f'{len(sources)} permitted enterprises across {sources.city.nunique()} cities and districts')
print()
print('Karaganda city:')
for name in sources[sources.city == 'Караганда'].source:
    print('  -', name)

Coal extraction, coal chemistry, ferroalloys and waste handling. But the episodes all occur at
posts in a residential district and involve particles rather than stack gases, which points at
domestic coal burning as the driver of the peaks and at permitted industry as the background.

## 8. Figures

All eight figures in the report are produced by `build.py`:

```bash
python build.py
```